# Spark on Apache Hadoop YARN: Architecture and Resource Allocation

**Module**: 12_advanced_topics  
**Notebook**: 01_spark_on_yarn.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark runs on Hadoop YARN: ResourceManager, NodeManager, ApplicationMaster, and yarn-client vs yarn-cluster modes.
- **Why does this topic exist?**: YARN is the dominant cluster manager in enterprise on-premises Hadoop data platforms.
- **What problem does it solve?**: Configuring YARN memory overhead and queue allocations.
- **Where is it used in real-world Data Engineering?**: Enterprise banking and telecom Hadoop clusters.

---

## 2. Spark on YARN Architecture
```
YARN Cluster:
  - ResourceManager (RM): Central cluster arbiter.
  - NodeManager (NM): Runs on each worker node, launching containers.
Application Submission:
  1. Client submits job to RM.
  2. RM allocates Container 0 for Spark ApplicationMaster (AM).
  3. In yarn-cluster mode, the AM hosts the Spark Driver!
  4. In yarn-client mode, Driver runs on client; AM only requests executors.
  5. AM requests executor containers across NodeManagers.
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("YARN_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("Cluster Manager in our Lab: Spark Standalone (spark://spark-master:7077)")
print("YARN submission equivalent command:")
print("spark-submit --master yarn --deploy-mode cluster --num-executors 10 --executor-memory 4g my_app.py")

---

## Experiment: Comparing Standalone vs YARN

- **1. What are we testing?**: Contrast Standalone Master with YARN ResourceManager.
- **2. What do we expect?**: Standalone is dedicated to Spark; YARN manages multi-tenant workloads (Hive, Flink, Spark, MapReduce).

In [ ]:
print('YARN provides multi-tenant queue scheduling (CapacityScheduler, FairScheduler).')

**3. What happened?**: Explained YARN architectural distinctions.  
**4. Why did it happen?**: YARN is a generic resource manager, not Spark-specific.  
**5. Practical Engineering Lesson**: Use Standalone for dedicated Spark clusters; use YARN for multi-tenant Hadoop clusters.

---

## Practical Exercises

### Exercise 1 (Beginner): YARN Deploy Modes
**Problem Statement**: What is the difference between yarn-client and yarn-cluster?  
**Input / Context**: Modes

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): YARN Memory Overhead Config
**Problem Statement**: What is spark.yarn.executor.memoryOverhead?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): CapacityScheduler Queues
**Problem Statement**: How to specify a YARN queue in spark-submit.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (YARN Deploy Modes):
```python
print('In yarn-client, Driver runs on client machine; in yarn-cluster, Driver runs inside YARN ApplicationMaster container.')
```

#### Solution 2 (YARN Memory Overhead Config):
```python
print('Off-heap container buffer for YARN (default max(384MB, 10% of heap)).')
```

#### Solution 3 (CapacityScheduler Queues):
```python
print('--queue production_etl')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **YARN AM**: ApplicationMaster negotiates containers.
- **Deploy Modes**: yarn-client for dev/Jupyter; yarn-cluster for production.

### Essential Technical Terms
- `ResourceManager`: Central YARN scheduler.
- `ApplicationMaster`: Per-application coordinator in YARN.

### Common Engineering Mistakes
- **yarn-client in production cron jobs**: If client terminal closes, job terminates.

### Interview Questions on This Topic
- **Q: Where does the Driver run in yarn-cluster mode?**
  - *A*: Inside the YARN ApplicationMaster container on a worker node in the cluster.

### What to Study Next
Open 02_spark_on_kubernetes.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')